# Capstone 3 — Session 3: Data Analysis with Pandas

**Run timestamp:** `2026-02-19 01:49:22`

## Goal
- Perform structured exploratory analysis on the Capstone 2 processed dataset using categorical profiling, pivot analysis, and distribution tables.
- Produce interpretable tables for health, region, demographic, and income patterns to support downstream business insights.

## Inputs
- `NSMES1988updated.csv` (primary source from `Capstone 2/outputs/`)

## Outputs
- All exports go to `./outputs/` (and plots to `./outputs/plots/` when applicable)

## Libraries (documented)
- `pandas`: needed for pivots, crosstabs, grouping, and distribution tables; enabled all required session analyses.

## Key dataset note
- `age` is encoded as **Age in years (divided by 10)** (e.g., `6.9` = 69 years).

## C?-T0 — Runtime setup (paths + output folders)
I used this setup block first to configure reproducible paths and output folders.

In [1]:
from pathlib import Path
from datetime import datetime

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

# --- Project metadata ---
CAPSTONE = 3
SESSION_TITLE = 'Session 3: Data Analysis with Pandas'

print(f"Capstone: {CAPSTONE} | Session: {SESSION_TITLE}")
print("Run timestamp:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

CWD = Path.cwd()
if (CWD / f"Capstone {CAPSTONE}").exists():
    BASE_DIR = CWD / f"Capstone {CAPSTONE}"
elif CWD.name == f"Capstone {CAPSTONE}":
    BASE_DIR = CWD
elif (CWD / "Incremental_Capstone" / f"Capstone {CAPSTONE}").exists():
    BASE_DIR = CWD / "Incremental_Capstone" / f"Capstone {CAPSTONE}"
else:
    BASE_DIR = CWD

# --- Paths ---
def first_existing_path(candidates):
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        if candidate.exists():
            return candidate
    return None

def resolve_dataset_path(default_filename: str) -> Path:
    """Resolve dataset path from known local runtime locations (non-interactive)."""
    path = first_existing_path([
        BASE_DIR / default_filename,
        CWD / default_filename,
        CWD / "Incremental_Capstone" / f"Capstone {CAPSTONE}" / default_filename,
    ])
    if path is None:
        path = BASE_DIR / default_filename

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    return path

OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Plots directory:", PLOTS_DIR)


Capstone: 3 | Session: Session 3: Data Analysis with Pandas
Run timestamp: 2026-02-19 00:31:46
Output directory: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 3\outputs
Plots directory: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 3\outputs\plots


## C?-T1 — Imports I used
I documented each import with why I used it and what it enabled in this analysis.

In [2]:
import pandas as pd  # DataFrames + pivot/crosstab/groupby analysis for categorical and distribution tables

## C?-T2 — Load dataset
I loaded the required Capstone 3 dataset with the configured default and fallback path.

In [3]:
DEFAULT_DATASET = "NSMES1988updated.csv"

try:
    dataset_path = resolve_dataset_path(DEFAULT_DATASET)
except FileNotFoundError:
    fallback = first_existing_path([
        BASE_DIR.parent / "Capstone 2" / "outputs" / "NSMES1988updated.csv",
        CWD / "Capstone 2" / "outputs" / "NSMES1988updated.csv",
        CWD / "Incremental_Capstone" / "Capstone 2" / "outputs" / "NSMES1988updated.csv",
    ])
    if fallback is None:
        raise
    dataset_path = fallback
    print("Local default not found; using fallback:", dataset_path)

df = pd.read_csv(dataset_path)

print("Loaded:", dataset_path)
print("Shape:", df.shape)
display(df.head())


Local default not found; using fallback: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 2\outputs\NSMES1988updated.csv
Loaded: c:\DEV_Projects\SIMPLILEARN\CAPSTONE_Applied_Data_Science_with _Python\Incremental_Capstone\Capstone 2\outputs\NSMES1988updated.csv
Shape: (4406, 20)


,visits,nvisits,ovisits,novisits,emergency,hospital,health,chronic,adl,region,age,gender,married,school,income,employed,insurance,medicaid,age_years,income_dollars
0,5,0,0,0,0,1,average,2,normal,other,6.9,male,yes,6,2.8810,yes,yes,no,69,28810
1,1,0,2,0,2,0,average,2,normal,other,7.4,female,yes,10,2.7478,no,yes,no,74,27478
2,13,0,0,0,3,3,poor,4,limited,other,6.6,female,no,10,0.6532,no,no,yes,66,6532
3,16,0,5,0,1,1,poor,2,limited,other,7.6,male,yes,3,0.6588,no,yes,no,76,6588
4,3,0,0,0,0,0,average,2,limited,other,7.9,female,yes,6,0.6588,no,yes,no,79,6588


## C?-T3 — Validation checks
- I confirmed expected columns exist
- I confirmed key dtypes
- I checked missing values

**Results Capture:**
- What I did: I validated expected Capstone 3 schema from Capstone 2 output, checked dtypes, and computed missing values.
- What I found: dataframe loaded as `(4406, 20)` with all expected columns including `age_years` and `income_dollars`; total missing values = 0.
- Caveats: label columns (`health`, `region`, `gender`, `married`, `employed`, `insurance`, `medicaid`) are categorical and should not be interpreted as continuous numeric values.

In [4]:
expected_cols = [
    "visits", "nvisits", "ovisits", "novisits", "emergency", "hospital",
    "health", "chronic", "adl", "region", "age", "gender",
    "married", "school", "income", "employed", "insurance", "medicaid",
    "age_years", "income_dollars"
]

missing_cols = [c for c in expected_cols if c not in df.columns]
print("Missing expected columns:", missing_cols)

print("\nDtypes:")
display(df.dtypes)

print("\nMissing values (count):")
na_counts = df.isna().sum().sort_values(ascending=False)
display(na_counts[na_counts > 0] if (na_counts > 0).any() else na_counts.head())


Missing expected columns: []

Dtypes:


visits              int64
nvisits             int64
ovisits             int64
novisits            int64
emergency           int64
hospital            int64
health                str
chronic             int64
adl                   str
region                str
age               float64
gender                str
married               str
school              int64
income            float64
employed              str
insurance             str
medicaid              str
age_years           int64
income_dollars      int64
dtype: object


Missing values (count):


visits       0
nvisits      0
ovisits      0
novisits     0
emergency    0
dtype: int64

## C3-T4 — Identify categorical types + key categories
**PDF requirement:** Identify different types of data and identify categorical types.

### What I completed
- I audited dtypes, grouped variables by type, and listed unique category values for health and region.

### Results Capture
- Built a type-audit table (`column`, `dtype`, `data_type_group`) for all 20 columns.
- `health` categories: `['average', 'excellent', 'poor']`.
- `region` categories: `['midwest', 'northeast', 'other', 'west']`.

### Code evidence
- The next cell shows the dtype audit and category-value extraction I executed.

In [5]:
categorical_cols = {"health", "adl", "region", "gender", "married", "employed", "insurance", "medicaid"}
dtype_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "data_type_group": ["categorical/flag" if c in categorical_cols else "numeric" for c in df.columns]
})
display(dtype_audit)

for col in ["health", "region"]:
    if col in df.columns:
        print(f"\nUnique values for {col}:")
        print(sorted(df[col].dropna().unique().tolist()))


,column,dtype,data_type_group
0,visits,int64,numeric
1,nvisits,int64,numeric
2,ovisits,int64,numeric
3,novisits,int64,numeric
4,emergency,int64,numeric
5,hospital,int64,numeric
6,health,str,categorical/flag
7,chronic,int64,numeric
8,adl,str,categorical/flag
9,region,str,categorical/flag



Unique values for health:
['average', 'excellent', 'poor']

Unique values for region:
['midwest', 'northeast', 'other', 'west']


## C3-T5 — Pivoting including Health and Region
**PDF requirement:** Perform detailed pivoting and include categorical data analysis for Health and Region.

### What I completed
- I generated pivot tables for counts, mean visits, and mean income by region and health.

### Results Capture
- Generated 3 detailed pivots by `region × health`: count matrix, mean visits, and mean income (dollars).
- Pivot count size = `(4, 3)` and total records represented = `4406`.
- Interpretation highlights:
  - All four regions contain observations across all three health categories.
  - Mean utilization varies by region-health combinations, supporting segmented analysis.
  - Mean income differences by region-health indicate socio-economic heterogeneity across segments.

### Code evidence
- The next cell contains the exact pivot computations and displayed outputs.

In [6]:
# Region-health pivots (counts + average visits + average income)
if set(["region", "health"]).issubset(df.columns):
    pivot_count = pd.pivot_table(df, index="region", columns="health", values="visits", aggfunc="count", fill_value=0)
    pivot_mean_visits = pd.pivot_table(df, index="region", columns="health", values="visits", aggfunc="mean", fill_value=0)
    pivot_mean_income = pd.pivot_table(df, index="region", columns="health", values="income_dollars", aggfunc="mean", fill_value=0)
    print("Pivot 1: Count by region x health")
    display(pivot_count)
    print("Pivot 2: Mean visits by region x health")
    display(pivot_mean_visits.round(3))
    print("Pivot 3: Mean income_dollars by region x health")
    display(pivot_mean_income.round(2))


Pivot 1: Count by region x health


health,average,excellent,poor
region,,,
midwest,957,90,110
northeast,694,57,86
other,1237,105,272
west,621,91,86


Pivot 2: Mean visits by region x health


health,average,excellent,poor
region,,,
midwest,5.282,3.444,8.118
northeast,5.695,4.018,10.674
other,5.088,3.143,8.746
west,6.499,3.374,8.593


Pivot 3: Mean income_dollars by region x health


health,average,excellent,poor
region,,,
midwest,24984.23,31053.89,21618.12
northeast,27015.35,33400.12,20659.30
other,22010.96,30024.58,16851.79
west,31663.77,37255.87,21118.85


## C3-T6 — Criteria-based analysis (visits, gender, marital, school, income, employed, insurance, medicaid)
**PDF requirement:** Analyze the data based on multiple criteria including visits type, gender, marital status, school, income, employed, insurance, medical aid.

### What I completed
- I produced grouped mean tables for each requested criterion across utilization and income metrics.

### Results Capture
- Produced one compact mean table per criterion (`gender`, `married`, `school`, `employed`, `insurance`, `medicaid`) for visit metrics and `income_dollars`.
- Key findings:
  - `gender`: mean visits `female=6.014` vs `male=5.420`; mean income `female=22,493.476`, `male=29,377.156`.
  - `insurance`: insured group has higher mean visits (`6.023`) than uninsured (`4.913`).
  - `medicaid`: enrollment corresponds to higher mean visits (`6.714`) but lower mean income (`11,365.388`) vs non-enrolled.

### Code evidence
- The next cell shows all criterion-based grouped tables I generated.

In [7]:
criteria_cols = ["gender", "married", "school", "employed", "insurance", "medicaid"]
metric_cols = [c for c in ["visits", "nvisits", "ovisits", "novisits", "emergency", "hospital"] if c in df.columns]

print("Metric columns guess:", metric_cols)

for c in criteria_cols:
    if c in df.columns and metric_cols:
        print(f"\nCriteria table: {c}")
        table = df.groupby(c)[metric_cols + ["income_dollars"]].mean(numeric_only=True).round(3)
        display(table)


Metric columns guess: ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital']

Criteria table: gender


,visits,nvisits,ovisits,novisits,emergency,hospital,income_dollars
gender,,,,,,,
female,6.014,1.732,0.683,0.513,0.269,0.286,22493.476
male,5.420,1.449,0.851,0.570,0.255,0.311,29377.156



Criteria table: married


,visits,nvisits,ovisits,novisits,emergency,hospital,income_dollars
married,,,,,,,
no,5.932,1.549,0.650,0.507,0.302,0.311,17194.294
yes,5.644,1.675,0.835,0.560,0.232,0.283,31985.391



Criteria table: school


...

,visits,nvisits,ovisits,novisits,emergency,hospital,income_dollars
employed,,,,,,,
no,5.807,1.640,0.786,0.549,0.269,0.304,23600.560
yes,5.495,1.426,0.444,0.424,0.220,0.226,39779.402



Criteria table: insurance


,visits,nvisits,ovisits,novisits,emergency,hospital,income_dollars
insurance,,,,,,,
no,4.913,0.934,1.056,0.436,0.333,0.311,16629.839
yes,6.023,1.815,0.663,0.565,0.243,0.292,27759.441



Criteria table: medicaid


,visits,nvisits,ovisits,novisits,emergency,hospital,income_dollars
medicaid,,,,,,,
no,5.680,1.654,0.725,0.534,0.246,0.284,26667.471
yes,6.714,1.259,1.007,0.555,0.443,0.415,11365.388


## C3-T7 — Create required distribution tables
**PDF requirement:** Create distribution tables: Age+Gender; Health by Gender; Income by Gender; Regional Income; Age-wise Income.

### What I completed
- I built all required distribution tables using age groups, crosstabs, and grouped income summaries.

### Results Capture
- Built all required distribution tables:
  - Age + Gender (`5x2`), Health by Gender (`3x2`), Income by Gender (`2x3`), Regional Income (`4x3`), Age-wise Income (`5x3`).
- Notable patterns:
  - Regional mean income is highest in `west` and lowest in `other`.
  - Male mean income exceeds female mean income, while female mean visits are higher.
  - Older age groups remain strongly represented across both genders.

### Code evidence
- The next cell contains the exact table-generation logic for all required distributions.

In [8]:
df3 = df.copy()

age_col = "age_years" if "age_years" in df3.columns else "age"
income_col = "income_dollars" if "income_dollars" in df3.columns else "income"

if age_col in df3.columns:
    df3["age_group"] = pd.cut(df3[age_col].astype(float), bins=[65, 70, 75, 80, 85, 110], include_lowest=True)

if income_col in df3.columns:
    quantiles = df3[income_col].quantile([0, .2, .4, .6, .8, 1.0]).values
    edges = [quantiles[0]]
    for value in quantiles[1:]:
        edges.append(value if value > edges[-1] else edges[-1] + 1)
    df3["income_band"] = pd.cut(df3[income_col].astype(float), bins=edges, include_lowest=True)

if set(["age_group", "gender"]).issubset(df3.columns):
    print("Distribution 1: Age + Gender")
    display(pd.crosstab(df3["age_group"], df3["gender"]))

if set(["health", "gender"]).issubset(df3.columns):
    print("Distribution 2: Health by Gender")
    display(pd.crosstab(df3["health"], df3["gender"]))

if set(["income_dollars", "gender"]).issubset(df3.columns):
    print("Distribution 3: Income by Gender")
    display(pd.pivot_table(df3, index="gender", values="income_dollars", aggfunc=["count", "mean", "median"]).round(2))

if set(["income_dollars", "region"]).issubset(df3.columns):
    print("Distribution 4: Regional Income")
    display(pd.pivot_table(df3, index="region", values="income_dollars", aggfunc=["count", "mean", "median"]).round(2))

if set(["age_group", "income_dollars"]).issubset(df3.columns):
    print("Distribution 5: Age-wise Income")
    display(pd.pivot_table(df3, index="age_group", values="income_dollars", aggfunc=["count", "mean", "median"]).round(2))


Distribution 1: Age + Gender


gender,female,male
age_group,,
"(64.999, 70.0]",897,671
"(70.0, 75.0]",736,530
"(75.0, 80.0]",525,321
"(80.0, 85.0]",303,176
"(85.0, 110.0]",167,80


Distribution 2: Health by Gender


gender,female,male
health,,
average,2093,1416
excellent,193,150
poor,342,212


Distribution 3: Income by Gender


,count,mean,median
,income_dollars,income_dollars,income_dollars
gender,,,
female,2628,22493.48,14160.0
male,1778,29377.16,20574.0


Distribution 4: Regional Income


,count,mean,median
,income_dollars,income_dollars,income_dollars
region,,,
midwest,1157,25136.34,17875.0
northeast,837,26797.09,17413.0
other,1614,21662.84,14220.0
west,798,31165.05,20656.0


Distribution 5: Age-wise Income


,count,mean,median
,income_dollars,income_dollars,income_dollars
age_group,,,
"(64.999, 70.0]",1568,27488.76,19872.0
"(70.0, 75.0]",1266,26194.35,16986.0
"(75.0, 80.0]",846,22997.60,15135.0
"(80.0, 85.0]",479,20269.47,12886.0
"(85.0, 110.0]",247,23951.36,13940.0


## Final section — Conclusions (required)
- I used `Capstone 2/outputs/NSMES1988updated.csv` as the input for this analysis.
- I completed categorical profiling and pivoting for health and region with clear segmentation outputs.
- I documented meaningful criteria-based differences across gender, insurance, medicaid, marital status, employment, and schooling groups.
- I generated and interpreted all required distribution tables.
- I updated `WORK_SUMMARY.md` with evidence and marked all Capstone 3 tasks complete.


In [9]:
print("Capstone 3 completed: C3-T4 to C3-T7")
print("Input used: Capstone 2/outputs/NSMES1988updated.csv")


Capstone 3 completed: C3-T4 to C3-T7
Input used: Capstone 2/outputs/NSMES1988updated.csv
